In [1]:
import dspy

print(dspy.__version__)

local_llm = dspy.LM(
    "openai/qwen3:30b", 
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed"
)

dspy.configure(lm=local_llm,  cache=False)
dspy.configure_cache(enable_disk_cache=False)
dspy.configure_cache(enable_memory_cache=False)

3.0.4


In [2]:
class GenerateSearchQuery(dspy.Signature):
    """Generiert eine Suchanfrage basierend auf einer Frage und dem bisherigen Kontext, um fehlende Informationen zu finden."""
    context = dspy.InputField(desc="Die bisher gesammelten Fakten.")
    question = dspy.InputField(desc="Die ursprüngliche Frage.")
    query = dspy.OutputField(desc="Eine präzise Suchanfrage für den nächsten Informationsschritt.")


class GenerateAnswerWithConflictCheck(dspy.Signature):
    """
    Beantwortet die Frage basierend auf dem gesammelten Kontext.
    Analysiere den Kontext auf Widersprüche. 
    Falls widersprüchliche Informationen vorliegen, weise explizit darauf hin und nenne die konkurrierenden Informationen.
    """
    context = dspy.InputField(desc="Alle relevanten Informationen aus mehreren Suchschritten.")
    question = dspy.InputField(desc="Die zu beantwortende Frage.")
    answer = dspy.OutputField(desc="Die finale Antwort, inklusive Hinweis auf eventuelle Widersprüche.")


In [3]:
class MultiHopRAG(dspy.Module):
    def __init__(self, max_hops=2):
        super().__init__()
        self.max_hops = max_hops

        self.generate_query = dspy.ChainOfThought(GenerateSearchQuery)
        self.retrieve = dspy.Retrieve(k=5) # K erhöht, um Wahrscheinlichkeit für Widersprüche im Kontext zu erhöhen
        self.generate_answer = dspy.ChainOfThought(GenerateAnswerWithConflictCheck)

    def forward(self, question):
        context = []

        for hop in range(self.max_hops):
            query_result = self.generate_query(context=context, question=question)
            search_query = query_result.query

            raw_passages = self.retrieve(search_query).passages
            passages = [psg for psg in raw_passages]

            # Kontext erweitern (Deduplizierung)
            context = list(set(context + passages))

        prediction = self.generate_answer(context=context, question=question)
        return prediction


In [4]:
# 1. Define a simple Retriever class
class LocalRetriever(dspy.Retrieve):
    def __init__(self, docs, k=3):
        super().__init__(k=k)
        self.docs = docs

    def forward(self, query_or_queries, k=None):
        k = k if k is not None else self.k
        query = query_or_queries if isinstance(query_or_queries, str) else query_or_queries[0]

        # Einfache Keyword-Suche
        results = [doc for doc in self.docs if any(word.lower() in doc.lower() for word in query.split())]

        # Fallback, falls keine Treffer
        if not results:
            results = self.docs

        predictions = [dspy.Example(long_text=doc) for doc in results[:k]]
        return predictions

# Erweiterte Dokumentenbasis mit Widersprüchen
docs = [
    "Der Gründer von Microsoft ist Bill Gates.",
    "Bill Gates wurde in den USA geboren.",
    "Die Hauptstadt der USA ist Washington, D.C..",
    # Widersprüchliche Informationen:
    "Bill Gates wurde in Frankreich geboren.",
    "Die Hauptstadt der USA ist New York City."
]

# 3. Instantiate your custom retriever
rm = LocalRetriever(docs)

# 4. Configure DSPy to use it
dspy.configure(rm=rm)

In [5]:
rm = LocalRetriever(docs)
dspy.configure(rm=rm)

multi_hop_program = MultiHopRAG(max_hops=2)

# Frage, die auf die widersprüchlichen Daten abzielt
question = "Wie heißt die Hauptstadt des Geburtslandes des Gründers von Microsoft?"

# Ausführung
response = multi_hop_program(question=question)

print(f"Frage: {question}")
print(f"Antwort: {response.answer}")

Frage: Wie heißt die Hauptstadt des Geburtslandes des Gründers von Microsoft?
Antwort: Der Kontext widerspricht sich: Bill Gates wurde entweder in den USA (Hauptstadt: Washington D.C.) oder in Frankreich (Hauptstadt: Paris) geboren. Aufgrund der widersprüchlichen Angaben [2] und [3] ist die Antwort nicht eindeutig.


In [6]:
# Optional: Inspektion der Gedankengänge
local_llm.inspect_history(n=10)





[2025-11-25T16:44:19.149615]

System message:

Your input fields are:
1. `context` (str): Die bisher gesammelten Fakten.
2. `question` (str): Die ursprüngliche Frage.
Your output fields are:
1. `reasoning` (str): 
2. `query` (str): Eine präzise Suchanfrage für den nächsten Informationsschritt.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## query ## ]]
{query}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Generiert eine Suchanfrage basierend auf einer Frage und dem bisherigen Kontext, um fehlende Informationen zu finden.


User message:

[[ ## context ## ]]
N/A

[[ ## question ## ]]
Wie heißt die Hauptstadt des Geburtslandes des Gründers von Microsoft?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## query ## ]]`, and then ending with t